
# CARBANAK process-tree preparation

This notebook prepares process-parent edges from a local `edr-events.tar.gz` archive.

Default input path, relative to this notebook directory/kernel working directory:

```text
../../data/CARBANAK/edr-events.tar.gz
```

Main edge output:

```text
processed/carbanak_process_edges.csv
```

with exactly:

```text
timestamp,parent_guid,process_guid
```

Additional sidecar outputs now include plausible process-name node labels:

```text
processed/carbanak_node_labels.csv
processed/carbanak_node_labels_subsample.csv
```

The node labels are short process names such as `cmd.exe`, not full paths, when the raw records contain usable process-name or process-path fields.


If the raw records contain boolean/string ground-truth fields such as `malicious` or `is_malicious`, the node-label sidecar also includes an `is_malicious` flag.

Ground-truth alert labels are optional. If `edr_alert_labels.xlsx` is present in `DATA_DIR`
(or the path in `CARBANAK_ALERT_LABELS_XLSX`), this notebook joins those labels by process GUID
and writes malicious/artifact/benign fields into the node-label sidecars.


In [1]:

from pathlib import Path
from collections import Counter, defaultdict
from zipfile import ZIP_DEFLATED
import bz2
import gzip
import io
import json
import lzma
import os
import pickle
import random
import re
import tarfile

import networkx as nx
import pandas as pd
from IPython.display import display

DATA_DIR = Path(os.environ.get("CARBANAK_DATA_DIR", "../../data/CARBANAK")).resolve()
ARCHIVE_PATH = DATA_DIR / "edr-events.tar.gz"
OUT_DIR = DATA_DIR / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PROCESS_EDGES_CSV = OUT_DIR / "carbanak_process_edges.csv"
SUBSAMPLE_EDGES_CSV = OUT_DIR / "carbanak_process_edges_subsample.csv"
SUBSAMPLE_GRAPH_PKL = OUT_DIR / "carbanak_process_graph_subsample.pkl"
COMPONENT_SUMMARY_CSV = OUT_DIR / "carbanak_component_summary.csv"
SUBSAMPLE_COMPONENTS_CSV = OUT_DIR / "carbanak_subsample_components.csv"
FULL_GRAPH_PKL = OUT_DIR / "carbanak_process_graph_full.pkl"

NODE_LABELS_CSV = OUT_DIR / "carbanak_node_labels.csv"
SUBSAMPLE_NODE_LABELS_CSV = OUT_DIR / "carbanak_node_labels_subsample.csv"
NODE_TIMES_CSV = OUT_DIR / "carbanak_node_times.csv"
SUBSAMPLE_NODE_TIMES_CSV = OUT_DIR / "carbanak_node_times_subsample.csv"

# Optional ground-truth alert labels. Put the uploaded spreadsheet here, or set
# CARBANAK_ALERT_LABELS_XLSX to an absolute path. The expected columns include:
# timestamp, guid, parent_guid, path, parent_path, report, severity, record_id, label.
ALERT_LABELS_XLSX = Path(os.environ.get("CARBANAK_ALERT_LABELS_XLSX", DATA_DIR / "edr_alert_labels.xlsx")).resolve()
ALERT_LABEL_RECORDS_CSV = OUT_DIR / "carbanak_external_alert_label_records.csv"

EXTRACTION_MEMBER_LOG_CSV = OUT_DIR / "carbanak_extraction_member_log.csv"
EXTRACTION_RULE_LOG_CSV = OUT_DIR / "carbanak_extraction_rule_log.csv"
SCHEMA_PROBE_CSV = OUT_DIR / "carbanak_schema_probe.csv"

OUTPUT_COLUMNS = ["timestamp", "parent_guid", "process_guid"]

# Explicit source-field candidates. Edit these if the schema probe shows a different field name.
TIMESTAMP_CANDIDATES = [
    "timestamp",
    "event_timestamp",
    "backend_timestamp",
    "backend_update_timestamp",
    "device_timestamp",
    "process_start_time",
    "process_create_time",
    "start",
    "create_time",
    "created_at",
    "first_event_timestamp",
    "last_event_timestamp",
    "detection_timestamp",
]
PARENT_GUID_CANDIDATES = [
    "parent_guid",
    "parent_process_guid",
    "parent_process_unique_id",
    "parent_process_uuid",
]
PROCESS_GUID_CANDIDATES = [
    "process_guid",
    "process_unique_id",
    "process_uuid",
]
CHILD_PROCESS_GUID_CANDIDATES = [
    "child_process_guid",
    "childproc_guid",
    "childproc_process_guid",
    "child_guid",
    "child_unique_id",
]

# Label/name candidates. The notebook stores a short basename such as "cmd.exe".
PROCESS_NAME_CANDIDATES = [
    "process_name",
    "process_filename",
    "process_image",
    "process_image_name",
    "process_path",
    "process_filepath",
    "process_executable",
    "process_cmdline",
    "process_command_line",
    "command_line",
    "cmdline",
    "path",
    "image_name",
]
PARENT_PROCESS_NAME_CANDIDATES = [
    "parent_process_name",
    "parent_name",
    "parent_process_filename",
    "parent_process_image",
    "parent_process_image_name",
    "parent_process_path",
    "parent_path",
    "parent_process_filepath",
    "parent_process_executable",
    "parent_process_cmdline",
    "parent_process_command_line",
    "parent_cmdline",
    "parent_command_line",
]
CHILD_PROCESS_NAME_CANDIDATES = [
    "child_process_name",
    "childproc_name",
    "child_process_filename",
    "child_process_image",
    "childproc_image",
    "child_process_path",
    "childproc_path",
    "child_process_filepath",
    "child_process_executable",
    "child_process_cmdline",
    "childproc_cmdline",
    "child_process_command_line",
]

# Optional malicious/ground-truth candidates. If a schema probe shows a better
# field, add it here. Record-level candidates are assigned to the process/child
# endpoint, not to the parent endpoint.
PROCESS_MALICIOUS_CANDIDATES = [
    "process_is_malicious",
    "is_malicious",
    "malicious",
    "process_malicious",
    "ground_truth_malicious",
    "ground_truth",
    "groundtruth",
    "label",
    "classification",
    "verdict",
    "threat_label",
]
PARENT_MALICIOUS_CANDIDATES = [
    "parent_is_malicious",
    "parent_malicious",
    "parent_ground_truth",
    "parent_label",
    "parent_classification",
]
CHILD_MALICIOUS_CANDIDATES = [
    "child_is_malicious",
    "child_malicious",
    "child_process_is_malicious",
    "child_process_malicious",
    "child_ground_truth",
    "child_label",
    "child_classification",
]

ENABLE_DIRECT_PARENT_PROCESS_RULE = True
ENABLE_CHILDPROC_RULE = True

REQUIRE_AT_LEAST_ONE_OUTPUT_ROW = True
CHUNKSIZE = 200_000
DROP_SELF_LOOPS = True
DEDUPLICATE_OUTPUT_ROWS = True
NORMALIZE_GUIDS = True

SCHEMA_PROBE_ROWS_PER_MEMBER = 2_000

# Subsample controls.
# The default intentionally keeps many of the largest connected components,
# rather than stopping after a single huge component.
MIN_COMPONENT_NODES_FOR_SUBSAMPLE = 3
MIN_SUBSAMPLE_COMPONENTS = 100
MAX_SUBSAMPLE_COMPONENTS = 500
REQUIRE_MIN_SUBSAMPLE_COMPONENTS = True
TARGET_SUBSAMPLE_BYTES = 50 * 1024 * 1024
COMPONENT_ORDER = "largest_first"  # "largest_first" or "random"
RANDOM_SEED = 0

# If external malicious labels are present, prefer components containing them
# when choosing the subsample. This makes later EDA notebooks much more useful.
PREFER_MALICIOUS_COMPONENTS = True

SAVE_FULL_GRAPH = False

print("DATA_DIR:", DATA_DIR)
print("ARCHIVE_PATH:", ARCHIVE_PATH)
print("OUT_DIR:", OUT_DIR)
print("ALERT_LABELS_XLSX:", ALERT_LABELS_XLSX, "(exists)" if ALERT_LABELS_XLSX.exists() else "(not found)")
if not ARCHIVE_PATH.exists():
    raise FileNotFoundError(f"Expected archive not found: {ARCHIVE_PATH}")


DATA_DIR: /home/asmi28/python_code/data/CARBANAK
ARCHIVE_PATH: /home/asmi28/python_code/data/CARBANAK/edr-events.tar.gz
OUT_DIR: /home/asmi28/python_code/data/CARBANAK/processed
ALERT_LABELS_XLSX: /home/asmi28/python_code/data/CARBANAK/edr_alert_labels.xlsx (exists)


## Archive and extraction helpers

In [2]:

def member_kind(name: str) -> str | None:
    lower = name.lower()
    if lower.endswith((".csv", ".csv.gz", ".csv.bz2", ".csv.xz")):
        return "csv"
    if lower.endswith((".tsv", ".tsv.gz", ".tsv.bz2", ".tsv.xz")):
        return "tsv"
    if lower.endswith((
        ".jsonl", ".jsonl.gz", ".jsonl.bz2", ".jsonl.xz",
        ".ndjson", ".ndjson.gz", ".ndjson.bz2", ".ndjson.xz",
        ".json", ".json.gz", ".json.bz2", ".json.xz",
    )):
        return "jsonl"
    return None


def open_member_text(tar: tarfile.TarFile, member: tarfile.TarInfo):
    raw = tar.extractfile(member)
    if raw is None:
        raise RuntimeError(f"Could not open archive member: {member.name}")
    lower = member.name.lower()
    stream = raw
    if lower.endswith(".gz"):
        stream = gzip.GzipFile(fileobj=raw)
    elif lower.endswith(".bz2"):
        stream = bz2.BZ2File(raw)
    elif lower.endswith((".xz", ".lzma")):
        stream = lzma.LZMAFile(raw)
    return io.TextIOWrapper(stream, encoding="utf-8", errors="replace")


def iter_flat_records(obj):
    """Yield dict records from common JSON/JSONL shapes."""
    if isinstance(obj, dict):
        yielded_nested = False
        for key in ("events", "results", "data", "records"):
            value = obj.get(key)
            if isinstance(value, list):
                for item in value:
                    if isinstance(item, dict):
                        yielded_nested = True
                        yield item
        if not yielded_nested:
            yield obj
    elif isinstance(obj, list):
        for item in obj:
            if isinstance(item, dict):
                yield item


def scalar_ok(value) -> bool:
    if value is None:
        return False
    try:
        if pd.isna(value):
            return False
    except Exception:
        pass
    if isinstance(value, (dict, list, tuple, set)):
        return False
    return str(value).strip() != ""


def first_present(record: dict, keys: list[str]):
    for key in keys:
        if key in record and scalar_ok(record[key]):
            return key, record[key]
    return None, None


def canonical_guid(value):
    if not scalar_ok(value):
        return None
    s = str(value).strip()
    if s.lower() in {"none", "null", "nan", "<na>"}:
        return None
    # Normalize enough to join parent and process GUIDs without destroying structure.
    s = s.strip().strip("{}").strip()
    if NORMALIZE_GUIDS:
        s = s.lower()
    return s or None


_PROCESS_BASENAME_RE = re.compile(
    r"(?i)([A-Za-z0-9_.~$@#%+=(){}\[\]-]+\.(?:exe|com|bat|cmd|ps1|vbs|js|jse|wsf|scr|msi|dll|sys|tmp|bin))"
)


def short_process_name(value):
    """Return a short process label such as 'cmd.exe' from a name/path/cmdline-like field."""
    if not scalar_ok(value):
        return None
    s = str(value).strip().strip("'\"")
    if not s or s.lower() in {"none", "null", "nan", "<na>"}:
        return None

    # If there is an executable-looking token anywhere in the path/cmdline, use its basename.
    normalized = s.replace("\\", "/")
    matches = _PROCESS_BASENAME_RE.findall(normalized)
    if matches:
        return matches[-1].lower()

    # Otherwise, if the field looks path-like, take the last path component.
    if "/" in normalized:
        tail = normalized.rsplit("/", 1)[-1].strip("'\" ")
        if tail:
            return tail.split()[0].lower()

    # Last resort: first whitespace-delimited token.
    return s.split()[0].lower()


def first_process_label(record: dict, keys: list[str]):
    for key in keys:
        if key in record and scalar_ok(record[key]):
            label = short_process_name(record[key])
            if label:
                return key, label
    return None, None


def add_label_vote(label_votes: dict, guid, label, source_key: str | None):
    guid = canonical_guid(guid)
    if guid is None or label is None:
        return
    label_votes[guid][label] += 1
    if source_key is not None:
        label_source_votes[(guid, label)][source_key] += 1


def parse_malicious_value(value):
    """Return True/False/None from a raw ground-truth-like field."""
    if not scalar_ok(value):
        return None
    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)):
        try:
            return bool(float(value) != 0.0)
        except Exception:
            return None
    s = str(value).strip().lower()
    if s in {"1", "true", "t", "yes", "y", "malicious", "malware", "attack", "attacker", "carbanak", "bad", "suspicious"}:
        return True
    if s in {"0", "false", "f", "no", "n", "benign", "clean", "normal", "good", "unknown", "none", "null", "nan"}:
        return False
    # Conservative substring fallback for mixed labels such as "malicious:carbanak".
    if any(tok in s for tok in ["malicious", "malware", "attack", "carbanak"]):
        return True
    if any(tok in s for tok in ["benign", "clean", "normal"]):
        return False
    return None


def first_malicious_flag(record: dict, keys: list[str]):
    for key in keys:
        if key in record and scalar_ok(record[key]):
            parsed = parse_malicious_value(record[key])
            if parsed is not None:
                return key, parsed
    return None, None


def add_malicious_vote(malicious_votes: dict, guid, is_malicious, source_key: str | None):
    guid = canonical_guid(guid)
    if guid is None or is_malicious is None:
        return
    malicious_votes[guid][bool(is_malicious)] += 1
    if source_key is not None:
        malicious_source_votes[(guid, bool(is_malicious))][source_key] += 1


## Inspect archive members and schema

In [3]:

with tarfile.open(ARCHIVE_PATH, "r:*") as tar:
    members = [m for m in tar.getmembers() if m.isfile()]

member_table = pd.DataFrame(
    {
        "name": [m.name for m in members],
        "size_bytes": [m.size for m in members],
    }
).sort_values("size_bytes", ascending=False)

print(f"Archive regular files: {len(member_table):,}")
display(member_table.head(25))


Archive regular files: 6


,name,size_bytes
4,edr-events/h1-events.jsonl,7441693202
0,edr-events/h4-events.jsonl,4994205427
1,edr-events/h3-events.jsonl,4500987871
2,edr-events/h2-events.jsonl,2287652509
3,edr-events/dc-events.jsonl,364797344
5,edr-events/fs-events.jsonl,156553549


In [4]:

ALL_CANDIDATE_KEYS = (
    TIMESTAMP_CANDIDATES
    + PARENT_GUID_CANDIDATES
    + PROCESS_GUID_CANDIDATES
    + CHILD_PROCESS_GUID_CANDIDATES
    + PROCESS_NAME_CANDIDATES
    + PARENT_PROCESS_NAME_CANDIDATES
    + CHILD_PROCESS_NAME_CANDIDATES
    + PROCESS_MALICIOUS_CANDIDATES
    + PARENT_MALICIOUS_CANDIDATES
    + CHILD_MALICIOUS_CANDIDATES
)


def probe_jsonl_member(tar: tarfile.TarFile, member: tarfile.TarInfo, limit: int) -> dict:
    key_counts = Counter()
    candidate_counts = Counter()
    first_keys = None
    first_record_preview = None
    n_records = 0

    with open_member_text(tar, member) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            for rec in iter_flat_records(obj):
                n_records += 1
                if first_keys is None:
                    first_keys = sorted(rec.keys())
                    first_record_preview = {k: rec.get(k) for k in first_keys[:40]}
                key_counts.update(rec.keys())
                for key in ALL_CANDIDATE_KEYS:
                    if key in rec and scalar_ok(rec[key]):
                        candidate_counts[key] += 1
                if n_records >= limit:
                    break
            if n_records >= limit:
                break

    return {
        "member": member.name,
        "kind": "jsonl",
        "sample_records": n_records,
        "first_keys": first_keys or [],
        "candidate_hit_counts": dict(candidate_counts),
        "top_keys": dict(key_counts.most_common(80)),
        "first_record_preview": first_record_preview or {},
    }


def probe_tabular_member(tar: tarfile.TarFile, member: tarfile.TarInfo, kind: str) -> dict:
    sep = "\t" if kind == "tsv" else ","
    with open_member_text(tar, member) as f:
        preview = pd.read_csv(f, sep=sep, nrows=5)
    candidate_counts = {key: int(key in preview.columns) for key in ALL_CANDIDATE_KEYS}
    return {
        "member": member.name,
        "kind": kind,
        "sample_records": len(preview),
        "first_keys": list(preview.columns),
        "candidate_hit_counts": candidate_counts,
        "top_keys": {str(c): 1 for c in preview.columns[:80]},
        "first_record_preview": preview.head(1).to_dict(orient="records")[0] if len(preview) else {},
    }


probe_rows = []
with tarfile.open(ARCHIVE_PATH, "r:*") as tar:
    for member in members:
        kind = member_kind(member.name)
        if kind is None:
            continue
        if kind in {"csv", "tsv"}:
            row = probe_tabular_member(tar, member, kind)
        else:
            row = probe_jsonl_member(tar, member, SCHEMA_PROBE_ROWS_PER_MEMBER)
        probe_rows.append(row)

if not probe_rows:
    raise RuntimeError("No supported .csv/.tsv/.json/.jsonl/.ndjson members found in the archive.")

probe_df = pd.DataFrame(probe_rows)
probe_df.to_csv(SCHEMA_PROBE_CSV, index=False)

with pd.option_context("display.max_colwidth", None, "display.max_rows", 20):
    display(probe_df[["member", "kind", "sample_records", "candidate_hit_counts", "first_keys"]])

print("Saved schema probe to:", SCHEMA_PROBE_CSV)


,member,kind,sample_records,candidate_hit_counts,first_keys
0,edr-events/h4-events.jsonl,jsonl,2000,"{'backend_timestamp': 2000, 'device_timestamp': 2000, 'parent_guid': 2000, 'process_guid': 2000, 'process_path': 2000, 'process_cmdline': 2000, 'parent_path': 2000, 'parent_cmdline': 1996}","[action, backend_timestamp, device_external_ip, device_id, device_name, device_os, device_timestamp, event_origin, filemod_name, org_key, parent_cmdline, parent_guid, parent_hash, parent_path, parent_pid, parent_reputation, process_cmdline, process_guid, process_hash, process_path, process_pid, process_publisher, process_reputation, process_terminated, process_username, schema, sensor_action, type, version]"
1,edr-events/h3-events.jsonl,jsonl,2000,"{'backend_timestamp': 2000, 'device_timestamp': 2000, 'parent_guid': 1998, 'process_guid': 2000, 'process_path': 2000, 'process_cmdline': 1998, 'parent_path': 1998, 'parent_cmdline': 1998, 'childproc_guid': 5, 'childproc_name': 5}","[action, backend_timestamp, device_external_ip, device_id, device_name, device_os, device_timestamp, event_origin, filemod_name, org_key, parent_cmdline, parent_guid, parent_hash, parent_path, parent_pid, parent_reputation, process_cmdline, process_guid, process_hash, process_path, process_pid, process_publisher, process_reputation, process_terminated, process_username, schema, sensor_action, type, version]"
2,edr-events/h2-events.jsonl,jsonl,2000,"{'backend_timestamp': 2000, 'device_timestamp': 2000, 'parent_guid': 1997, 'process_guid': 2000, 'process_path': 2000, 'process_cmdline': 1997, 'parent_path': 1997, 'parent_cmdline': 1997, 'childproc_guid': 36, 'childproc_name': 36}","[action, backend_timestamp, device_external_ip, device_id, device_name, device_os, device_timestamp, event_origin, filemod_name, org_key, parent_cmdline, parent_guid, parent_hash, parent_path, parent_pid, parent_reputation, process_cmdline, process_guid, process_hash, process_path, process_pid, process_publisher, process_reputation, process_terminated, process_username, schema, sensor_action, type, version]"
3,edr-events/dc-events.jsonl,jsonl,2000,"{'backend_timestamp': 2000, 'device_timestamp': 2000, 'parent_guid': 1998, 'process_guid': 2000, 'process_path': 2000, 'process_cmdline': 1998, 'parent_path': 1998, 'parent_cmdline': 1975, 'childproc_guid': 22, 'childproc_name': 22}","[action, backend_timestamp, device_external_ip, device_id, device_name, device_os, device_timestamp, event_description, event_id, event_origin, local_ip, local_port, netconn_domain, netconn_inbound, netconn_protocol, org_key, parent_guid, parent_hash, parent_path, parent_pid, parent_reputation, process_cmdline, process_guid, process_hash, process_path, process_pid, process_publisher, process_reputation, process_terminated, process_username, remote_ip, remote_port, schema, sensor_action, type, version]"
4,edr-events/h1-events.jsonl,jsonl,2000,"{'backend_timestamp': 2000, 'device_timestamp': 2000, 'parent_guid': 1935, 'process_guid': 2000, 'process_path': 2000, 'process_cmdline': 1935, 'parent_path': 1935, 'parent_cmdline': 1935, 'childproc_guid': 31, 'childproc_name': 31}","[action, backend_timestamp, device_external_ip, device_id, device_name, device_os, device_timestamp, event_origin, modload_count, modload_effective_reputation, modload_hash, modload_md5, modload_name, modload_publisher, modload_sha256, org_key, parent_cmdline, parent_guid, parent_hash, parent_path, parent_pid, parent_reputation, process_cmdline, process_guid, process_hash, process_path, process_pid, process_publisher, process_reputation, process_terminated, process_username, schema, sensor_action, type, version]"
5,edr-events/fs-events.jsonl,jsonl,2000,"{'backend_timestamp': 2000, 'device_timestamp': 2000, 'parent_guid': 1444, 'process_guid': 2000, 'childproc_guid': 1847, 'process_path': 2000, 'process_cmdline': 1917, 'parent_path': 1444, 'childproc_name': 1847, 'parent_cmdline': 150}","[action, backend_timestamp, childproc_guid, childproc_hash, chil

Saved schema probe to: /home/asmi28/python_code/data/CARBANAK/processed/carbanak_schema_probe.csv


## Extract edges and process-name labels

In [5]:

def extract_rows_from_record(record: dict, label_votes: dict, malicious_votes: dict):
    rows = []
    ts_key, timestamp = first_present(record, TIMESTAMP_CANDIDATES)
    if timestamp is None:
        return rows

    if ENABLE_DIRECT_PARENT_PROCESS_RULE:
        parent_key, parent_guid_raw = first_present(record, PARENT_GUID_CANDIDATES)
        process_key, process_guid_raw = first_present(record, PROCESS_GUID_CANDIDATES)
        parent_guid = canonical_guid(parent_guid_raw)
        process_guid = canonical_guid(process_guid_raw)
        if parent_guid is not None and process_guid is not None:
            rows.append({
                "timestamp": timestamp,
                "parent_guid": parent_guid,
                "process_guid": process_guid,
                "_rule": "direct_parent_process",
                "_timestamp_key": ts_key,
                "_parent_key": parent_key,
                "_process_key": process_key,
            })
            parent_label_key, parent_label = first_process_label(record, PARENT_PROCESS_NAME_CANDIDATES)
            process_label_key, process_label = first_process_label(record, PROCESS_NAME_CANDIDATES)
            add_label_vote(label_votes, parent_guid, parent_label, parent_label_key)
            add_label_vote(label_votes, process_guid, process_label, process_label_key)
            parent_mal_key, parent_mal = first_malicious_flag(record, PARENT_MALICIOUS_CANDIDATES)
            process_mal_key, process_mal = first_malicious_flag(record, PROCESS_MALICIOUS_CANDIDATES)
            add_malicious_vote(malicious_votes, parent_guid, parent_mal, parent_mal_key)
            add_malicious_vote(malicious_votes, process_guid, process_mal, process_mal_key)

    if ENABLE_CHILDPROC_RULE:
        parent_key, parent_guid_raw = first_present(record, PROCESS_GUID_CANDIDATES)
        child_key, child_guid_raw = first_present(record, CHILD_PROCESS_GUID_CANDIDATES)
        parent_guid = canonical_guid(parent_guid_raw)
        child_guid = canonical_guid(child_guid_raw)
        if parent_guid is not None and child_guid is not None:
            rows.append({
                "timestamp": timestamp,
                "parent_guid": parent_guid,
                "process_guid": child_guid,
                "_rule": "childproc",
                "_timestamp_key": ts_key,
                "_parent_key": parent_key,
                "_process_key": child_key,
            })
            parent_label_key, parent_label = first_process_label(record, PROCESS_NAME_CANDIDATES)
            child_label_key, child_label = first_process_label(record, CHILD_PROCESS_NAME_CANDIDATES)
            add_label_vote(label_votes, parent_guid, parent_label, parent_label_key)
            add_label_vote(label_votes, child_guid, child_label, child_label_key)
            parent_mal_key, parent_mal = first_malicious_flag(record, PROCESS_MALICIOUS_CANDIDATES)
            child_mal_key, child_mal = first_malicious_flag(record, CHILD_MALICIOUS_CANDIDATES + PROCESS_MALICIOUS_CANDIDATES)
            add_malicious_vote(malicious_votes, parent_guid, parent_mal, parent_mal_key)
            add_malicious_vote(malicious_votes, child_guid, child_mal, child_mal_key)

    if DROP_SELF_LOOPS:
        rows = [r for r in rows if str(r["parent_guid"]) != str(r["process_guid"])]
    return rows


def clean_output_frame(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame(columns=OUTPUT_COLUMNS)
    out = frame[OUTPUT_COLUMNS].dropna(subset=OUTPUT_COLUMNS).copy()
    out["parent_guid"] = out["parent_guid"].map(canonical_guid).astype("string")
    out["process_guid"] = out["process_guid"].map(canonical_guid).astype("string")
    out = out.dropna(subset=["parent_guid", "process_guid"])
    out = out[(out["parent_guid"].str.len() > 0) & (out["process_guid"].str.len() > 0)]
    if DROP_SELF_LOOPS:
        out = out[out["parent_guid"] != out["process_guid"]]
    return out[OUTPUT_COLUMNS]


def append_output_rows(frame: pd.DataFrame) -> int:
    if frame.empty:
        return 0
    write_header = not PROCESS_EDGES_CSV.exists()
    frame.to_csv(PROCESS_EDGES_CSV, mode="a", header=write_header, index=False)
    return len(frame)


def update_rule_counts(rule_counts: Counter, rows: list[dict]):
    for row in rows:
        rule_counts[(row.get("_rule"), row.get("_timestamp_key"), row.get("_parent_key"), row.get("_process_key"))] += 1


for path in [PROCESS_EDGES_CSV, NODE_LABELS_CSV, NODE_TIMES_CSV]:
    if path.exists():
        path.unlink()

member_logs = []
rule_counts = Counter()
label_votes = defaultdict(Counter)
label_source_votes = defaultdict(Counter)
malicious_votes = defaultdict(Counter)
malicious_source_votes = defaultdict(Counter)
total_rows = 0
seen_any_supported_member = False

with tarfile.open(ARCHIVE_PATH, "r:*") as tar:
    for member in members:
        kind = member_kind(member.name)
        if kind is None:
            member_logs.append({"member": member.name, "kind": None, "rows_written": 0, "status": "skipped_unsupported_suffix"})
            continue
        seen_any_supported_member = True
        rows_before = total_rows
        batch = []

        with open_member_text(tar, member) as f:
            if kind in {"csv", "tsv"}:
                sep = "\t" if kind == "tsv" else ","
                for chunk in pd.read_csv(f, sep=sep, chunksize=CHUNKSIZE):
                    for rec in chunk.to_dict(orient="records"):
                        rows = extract_rows_from_record(rec, label_votes, malicious_votes)
                        update_rule_counts(rule_counts, rows)
                        batch.extend(rows)
                        if len(batch) >= CHUNKSIZE:
                            total_rows += append_output_rows(clean_output_frame(pd.DataFrame(batch)))
                            batch.clear()
            else:
                for line_no, line in enumerate(f, start=1):
                    line = line.strip()
                    if not line:
                        continue
                    obj = json.loads(line)
                    if not isinstance(obj, dict):
                        raise RuntimeError(f"{member.name}:{line_no} is not a JSON object.")
                    for rec in iter_flat_records(obj):
                        rows = extract_rows_from_record(rec, label_votes, malicious_votes)
                        update_rule_counts(rule_counts, rows)
                        batch.extend(rows)
                        if len(batch) >= CHUNKSIZE:
                            total_rows += append_output_rows(clean_output_frame(pd.DataFrame(batch)))
                            batch.clear()

        if batch:
            total_rows += append_output_rows(clean_output_frame(pd.DataFrame(batch)))
            batch.clear()

        member_logs.append({
            "member": member.name,
            "kind": kind,
            "rows_written": total_rows - rows_before,
            "status": "processed",
        })

if not seen_any_supported_member:
    raise RuntimeError("The archive contained no supported data files: .csv, .tsv, .json, .jsonl, or .ndjson.")

log_df = pd.DataFrame(member_logs)
log_df.to_csv(EXTRACTION_MEMBER_LOG_CSV, index=False)

rule_log_df = pd.DataFrame([
    {
        "rule": key[0],
        "timestamp_key": key[1],
        "parent_key": key[2],
        "process_key": key[3],
        "rows": value,
    }
    for key, value in rule_counts.items()
]).sort_values("rows", ascending=False) if rule_counts else pd.DataFrame(columns=["rule", "timestamp_key", "parent_key", "process_key", "rows"])
rule_log_df.to_csv(EXTRACTION_RULE_LOG_CSV, index=False)

if REQUIRE_AT_LEAST_ONE_OUTPUT_ROW and total_rows == 0:
    print("No process-parent rows were found. Candidate-field probe:")
    with pd.option_context("display.max_colwidth", None, "display.max_rows", 20):
        display(probe_df[["member", "sample_records", "candidate_hit_counts", "first_keys"]])
    print("Member extraction log:")
    display(log_df)
    raise RuntimeError(
        "No process-parent rows were found. Inspect the schema probe above and edit candidate field lists."
    )

if DEDUPLICATE_OUTPUT_ROWS and total_rows > 0:
    tmp = pd.read_csv(PROCESS_EDGES_CSV, dtype={"parent_guid": "string", "process_guid": "string"})
    tmp["parent_guid"] = tmp["parent_guid"].map(canonical_guid).astype("string")
    tmp["process_guid"] = tmp["process_guid"].map(canonical_guid).astype("string")
    before = len(tmp)
    tmp = tmp.drop_duplicates(subset=OUTPUT_COLUMNS)
    tmp.to_csv(PROCESS_EDGES_CSV, index=False)
    total_rows = len(tmp)
    print(f"Deduplicated rows: {before:,} -> {total_rows:,}")

print(f"Wrote {total_rows:,} rows to {PROCESS_EDGES_CSV}")
print(f"CSV size: {PROCESS_EDGES_CSV.stat().st_size / (1024**2):.2f} MB")
print("Member log:", EXTRACTION_MEMBER_LOG_CSV)
print("Rule log:", EXTRACTION_RULE_LOG_CSV)

display(log_df[log_df["rows_written"] > 0].head(20))
display(rule_log_df.head(20))


Deduplicated rows: 9,888,262 -> 267,318
Wrote 267,318 rows to /home/asmi28/python_code/data/CARBANAK/processed/carbanak_process_edges.csv
CSV size: 34.16 MB
Member log: /home/asmi28/python_code/data/CARBANAK/processed/carbanak_extraction_member_log.csv
Rule log: /home/asmi28/python_code/data/CARBANAK/processed/carbanak_extraction_rule_log.csv


,member,kind,rows_written,status
0,edr-events/h4-events.jsonl,jsonl,2675369,processed
1,edr-events/h3-events.jsonl,jsonl,2465625,processed
2,edr-events/h2-events.jsonl,jsonl,1224958,processed
3,edr-events/dc-events.jsonl,jsonl,216071,processed
4,edr-events/h1-events.jsonl,jsonl,3127807,processed
5,edr-events/fs-events.jsonl,jsonl,178432,processed


,rule,timestamp_key,parent_key,process_key,rows
0,direct_parent_process,backend_timestamp,parent_guid,process_guid,9707107
1,childproc,backend_timestamp,process_guid,childproc_guid,181155


## Optional: join external ground-truth alert labels

If `edr_alert_labels.xlsx` is available, this cell joins its labels by `guid`.
The raw CARBANAK event archive may not contain these labels, so this sidecar is the authoritative
source for `malicious` / `artifact` / `benign` annotations used by the EDA notebooks.


In [6]:

ALERT_LABEL_PRIORITY = {"benign": 0, "artifact": 1, "malicious": 2}
ALERT_LABEL_VALUES = tuple(ALERT_LABEL_PRIORITY.keys())


def normalize_alert_label(value):
    if not scalar_ok(value):
        return None
    s = str(value).strip().lower()
    if not s or s in {"none", "null", "nan", "<na>", "unknown"}:
        return None
    if "malicious" in s or "malware" in s or "attack" in s:
        return "malicious"
    if "artifact" in s:
        return "artifact"
    if "benign" in s or "clean" in s or "normal" in s:
        return "benign"
    return s


def worst_alert_label(counter: Counter) -> str | None:
    if not counter:
        return None
    # Worst label wins; ties are broken by count.
    return max(counter, key=lambda label: (ALERT_LABEL_PRIORITY.get(label, -1), counter[label]))


external_alert_votes = defaultdict(Counter)
external_alert_report_votes = defaultdict(Counter)
external_alert_severity_votes = defaultdict(Counter)

if ALERT_LABELS_XLSX.exists():
    alert_df = pd.read_excel(ALERT_LABELS_XLSX)
    alert_df.columns = [str(c).strip() for c in alert_df.columns]
    required = {"guid", "label"}
    missing = required - set(alert_df.columns)
    if missing:
        raise RuntimeError(f"{ALERT_LABELS_XLSX} is missing required columns: {sorted(missing)}")

    rows = []
    for rec in alert_df.to_dict(orient="records"):
        guid = canonical_guid(rec.get("guid"))
        if guid is None:
            continue
        alert_label = normalize_alert_label(rec.get("label"))
        if alert_label is None:
            continue

        external_alert_votes[guid][alert_label] += 1
        if scalar_ok(rec.get("report")):
            external_alert_report_votes[guid][str(rec.get("report"))] += 1
        if scalar_ok(rec.get("severity")):
            external_alert_severity_votes[guid][str(rec.get("severity"))] += 1

        # These are ground-truth votes for the process GUID. Treat artifact as
        # non-malicious for the binary malicious flag, but preserve it separately
        # as a non-benign alert label.
        if alert_label == "malicious":
            add_malicious_vote(malicious_votes, guid, True, "external_alert_label")
        elif alert_label == "benign":
            add_malicious_vote(malicious_votes, guid, False, "external_alert_label")
        elif alert_label == "artifact":
            add_malicious_vote(malicious_votes, guid, False, "external_alert_label_artifact")

        # The alert spreadsheet is also a useful source of short process names.
        process_label_key, process_label = first_process_label(rec, ["path", "cmdline"])
        add_label_vote(label_votes, guid, process_label, f"external_alert:{process_label_key}" if process_label_key else None)

        parent_guid = canonical_guid(rec.get("parent_guid"))
        if parent_guid is not None:
            parent_label_key, parent_label = first_process_label(rec, ["parent_path", "parent_cmdline"])
            add_label_vote(label_votes, parent_guid, parent_label, f"external_alert:{parent_label_key}" if parent_label_key else None)

        rows.append({
            "process_guid": guid,
            "parent_guid": parent_guid,
            "alert_label": alert_label,
            "timestamp": rec.get("timestamp"),
            "path": rec.get("path"),
            "parent_path": rec.get("parent_path"),
            "report": rec.get("report"),
            "severity": rec.get("severity"),
            "record_id": rec.get("record_id"),
        })

    external_alert_df = pd.DataFrame(rows)
    external_alert_df.to_csv(ALERT_LABEL_RECORDS_CSV, index=False)

    print(f"Loaded external alert labels: {len(external_alert_df):,} rows -> {ALERT_LABEL_RECORDS_CSV}")
    print("External alert label counts:")
    display(external_alert_df["alert_label"].value_counts().rename_axis("alert_label").reset_index(name="n_rows"))
    print("Unique labelled process GUIDs:", len(external_alert_votes))
    print("Worst-label process counts:")
    worst_counts = Counter(worst_alert_label(v) for v in external_alert_votes.values())
    display(pd.DataFrame({"alert_label": list(worst_counts), "n_processes": list(worst_counts.values())}).sort_values("n_processes", ascending=False))
else:
    external_alert_df = pd.DataFrame(columns=["process_guid", "parent_guid", "alert_label", "timestamp", "path", "parent_path", "report", "severity", "record_id"])
    print("No external alert label spreadsheet found. Continuing without ground-truth malicious/artifact labels.")
    print("To enable it, put edr_alert_labels.xlsx at:", ALERT_LABELS_XLSX)


Loaded external alert labels: 13,702 rows -> /home/asmi28/python_code/data/CARBANAK/processed/carbanak_external_alert_label_records.csv
External alert label counts:


,alert_label,n_rows
0,benign,11194
1,malicious,1413
2,artifact,1095


Unique labelled process GUIDs: 7745
Worst-label process counts:


,alert_label,n_processes
0,benign,7330
1,artifact,252
2,malicious,163


## Build the directed NetworkX process graph

In [7]:

df = pd.read_csv(
    PROCESS_EDGES_CSV,
    usecols=OUTPUT_COLUMNS,
    dtype={"parent_guid": "string", "process_guid": "string"},
)

if df.empty:
    raise RuntimeError(f"Loaded zero rows from {PROCESS_EDGES_CSV}")

df["parent_guid"] = df["parent_guid"].map(canonical_guid).astype("string")
df["process_guid"] = df["process_guid"].map(canonical_guid).astype("string")

G = nx.from_pandas_edgelist(
    df,
    source="parent_guid",
    target="process_guid",
    create_using=nx.DiGraph()
)

print(f"Rows in process-edge CSV: {len(df):,}")
print(f"Directed graph nodes:      {G.number_of_nodes():,}")
print(f"Directed graph edges:      {G.number_of_edges():,}")
print(f"Weak components:           {nx.number_weakly_connected_components(G):,}")

weak_components = list(nx.weakly_connected_components(G))
component_edge_counts = []
for nodes in weak_components:
    sub = G.subgraph(nodes)
    component_edge_counts.append(sub.number_of_edges())

component_diag = pd.DataFrame({
    "n_nodes": [len(c) for c in weak_components],
    "n_edges": component_edge_counts,
})
component_diag["edges_minus_nodes_plus_one"] = component_diag["n_edges"] - component_diag["n_nodes"] + 1
print("Components with undirected cycle surplus > 0:", int((component_diag["edges_minus_nodes_plus_one"] > 0).sum()))

if SAVE_FULL_GRAPH:
    with open(FULL_GRAPH_PKL, "wb") as f:
        pickle.dump(G, f, protocol=pickle.HIGHEST_PROTOCOL)
    print("Saved full graph to:", FULL_GRAPH_PKL)


Rows in process-edge CSV: 267,318
Directed graph nodes:      117,333
Directed graph edges:      117,929
Weak components:           324
Components with undirected cycle surplus > 0: 74


## Save node labels and node times

In [8]:

# Best process-name label per node from all raw observations.
label_rows = []
for node in G.nodes():
    votes = label_votes.get(str(node), Counter())
    if votes:
        label, count = votes.most_common(1)[0]
        sources = ",".join(k for k, _ in label_source_votes.get((str(node), label), Counter()).most_common())
    else:
        label, count, sources = "unknown", 0, ""
    mal_votes = malicious_votes.get(str(node), Counter())
    malicious_true = int(mal_votes.get(True, 0))
    malicious_false = int(mal_votes.get(False, 0))

    alert_votes = external_alert_votes.get(str(node), Counter()) if "external_alert_votes" in globals() else Counter()
    alert_label = worst_alert_label(alert_votes) if "worst_alert_label" in globals() else None
    alert_label = alert_label or "unlabelled"
    alert_n_malicious = int(alert_votes.get("malicious", 0))
    alert_n_artifact = int(alert_votes.get("artifact", 0))
    alert_n_benign = int(alert_votes.get("benign", 0))
    alert_n_records = int(sum(alert_votes.values()))

    # Binary malicious is strict: only spreadsheet label == malicious, or raw
    # malicious votes if present, make this True. Artifact remains non-malicious
    # but is still non-benign and preserved as alert_label == "artifact".
    is_malicious = (alert_n_malicious > 0) or (malicious_true > malicious_false and malicious_true > 0)
    is_attack_label = alert_label in {"malicious", "artifact"} or bool(is_malicious)

    mal_sources = []
    for val in [True, False]:
        mal_sources.extend(k for k, _ in malicious_source_votes.get((str(node), val), Counter()).most_common())
    label_rows.append({
        "process_guid": str(node),
        "label": label,
        "process_name": label,
        "n_label_observations": int(count),
        "label_source_fields": sources,
        "is_malicious": bool(is_malicious),
        "is_attack_label": bool(is_attack_label),
        "alert_label": alert_label,
        "alert_n_records": alert_n_records,
        "alert_n_malicious": alert_n_malicious,
        "alert_n_artifact": alert_n_artifact,
        "alert_n_benign": alert_n_benign,
        "top_alert_reports": "; ".join(k for k, _ in (external_alert_report_votes.get(str(node), Counter()) if "external_alert_report_votes" in globals() else Counter()).most_common(5)),
        "top_alert_severities": "; ".join(k for k, _ in (external_alert_severity_votes.get(str(node), Counter()) if "external_alert_severity_votes" in globals() else Counter()).most_common(5)),
        "n_malicious_votes": malicious_true,
        "n_benign_votes": malicious_false,
        "malicious_source_fields": ",".join(dict.fromkeys(mal_sources)),
    })

node_label_df = pd.DataFrame(label_rows).sort_values(["n_label_observations", "process_guid"], ascending=[False, True])
node_label_df.to_csv(NODE_LABELS_CSV, index=False)
print(f"Saved node labels: {len(node_label_df):,} rows -> {NODE_LABELS_CSV}")
display(node_label_df.head(20))
print("Top labels:")
display(node_label_df["label"].value_counts().head(20).rename_axis("label").reset_index(name="n_nodes"))
print("Malicious label counts:")
display(node_label_df["is_malicious"].value_counts().rename_axis("is_malicious").reset_index(name="n_nodes"))
if "alert_label" in node_label_df.columns:
    print("External alert-label counts:")
    display(node_label_df["alert_label"].value_counts().rename_axis("alert_label").reset_index(name="n_nodes"))
    print("Attack-label counts:")
    display(node_label_df["is_attack_label"].value_counts().rename_axis("is_attack_label").reset_index(name="n_nodes"))

# Node time: earliest time as child if available; otherwise earliest outgoing event as parent.
def parse_timestamp_seconds(series: pd.Series) -> pd.Series:
    raw_numeric_time = pd.to_numeric(series, errors="coerce")
    if raw_numeric_time.notna().mean() > 0.95:
        med = float(raw_numeric_time.dropna().median())
        if med > 1e17:
            return raw_numeric_time / 1e9
        if med > 1e14:
            return raw_numeric_time / 1e6
        if med > 1e11:
            return raw_numeric_time / 1e3
        return raw_numeric_time

    s = series.astype("string").str.strip()
    s = s.str.replace(r"\s+UTC$", "", regex=True)
    s = s.str.replace(r"Z$", "+00:00", regex=True)
    parsed = pd.to_datetime(s, errors="coerce", utc=True)
    if parsed.isna().any():
        bad_examples = series[parsed.isna()].head(10).tolist()
        raise RuntimeError(f"Could not parse some timestamps. Examples: {bad_examples}")
    return parsed.astype("int64") / 1e9

df["_time_seconds"] = parse_timestamp_seconds(df["timestamp"])
child_time = df.groupby("process_guid")["_time_seconds"].min()
parent_out_time = df.groupby("parent_guid")["_time_seconds"].min()

time_rows = []
for node in G.nodes():
    if node in child_time.index:
        t = float(child_time.loc[node])
        source = "child_event"
    elif node in parent_out_time.index:
        t = float(parent_out_time.loc[node])
        source = "parent_outgoing_event"
    else:
        raise RuntimeError(f"No timestamp found for node {node!r}")
    time_rows.append({"process_guid": str(node), "time": t, "time_source": source})

node_time_df = pd.DataFrame(time_rows)
node_time_df.to_csv(NODE_TIMES_CSV, index=False)
print(f"Saved node times: {len(node_time_df):,} rows -> {NODE_TIMES_CSV}")
display(node_time_df.head(10))


Saved node labels: 117,333 rows -> /home/asmi28/python_code/data/CARBANAK/processed/carbanak_node_labels.csv


,process_guid,label,process_name,n_label_observations,label_source_fields,is_malicious,is_attack_label,alert_label,alert_n_records,alert_n_malicious,alert_n_artifact,alert_n_benign,top_alert_reports,top_alert_severities,n_malicious_votes,n_benign_votes,malicious_source_fields
88962,7dmf69pk-0c587426-000003b8-00000000-1da965a2b7...,svchost.exe,svchost.exe,533995,"parent_path,process_path,external_alert:parent...",False,False,unlabelled,0,0,0,0,,,0,0,
88963,7dmf69pk-0c587426-00000a64-00000000-1da965a2be...,tiworker.exe,tiworker.exe,518348,"process_path,parent_path,childproc_name",False,False,unlabelled,0,0,0,0,,,0,0,
42904,7dmf69pk-0c5454cf-000003a4-00000000-1da959c045...,svchost.exe,svchost.exe,488804,"parent_path,process_path,childproc_name",False,False,unlabelled,0,0,0,0,,,0,0,
43353,7dmf69pk-0c5454cf-00000180-00000000-1da959d7fa...,tiworker.exe,tiworker.exe,451586,"process_path,parent_path,childproc_name",False,False,unlabelled,0,0,0,0,,,0,0,
43470,7dmf69pk-0c5454cf-000003ac-00000000-1da9a401b4...,svchost.exe,svchost.exe,369318,"parent_path,process_path,external_alert:parent...",False,False,unlabelled,0,0,0,0,,,0,0,
96195,7dmf69pk-0c587426-000012b8-00000000-1da9b28171...,wscript.exe,wscript.exe,331672,"process_path,external_alert:path,external_aler...",True,True,malicious,818,818,0,0,Defense Evasion - Signed Script Proxy Executio...,0.1; 0.9; 1.0; 0.8; 0.7,818,0,external_alert_label
96193,7dmf69pk-0c587426-00001b78-00000000-1da9b28170...,cmd.exe,cmd.exe,331546,"parent_path,external_alert:parent_path,externa...",True,True,malicious,34,34,0,0,Defense Evasion - Suspicious Child Processes S...,0.4,34,0,external_alert_label
43582,7dmf69pk-0c5454cf-000013b8-00000000-1da9a40203...,tiworker.exe,tiworker.exe,315455,"process_path,childproc_name",False,False,unlabelled,0,0,0,0,,,0,0,
45647,7dmf69pk-0c5454cf-000012e0-00000000-1da9a8237e...,tiworker.exe,tiworker.exe,229655,"parent_path,process_path,external_alert:parent...",False,False,benign,1,0,0,1,Defense Evasion - Registry modification in HKL...,0.2,0,1,external_alert_label
87527,7dmf69pk-0c587426-000003c4-00000000-1da95922c3...,svchost.exe,svchost.exe,200937,"parent_path,process_path,childproc_name",False,False,unlabelled,0,0,0,0,,,0,0,


Top labels:


,label,n_nodes
0,msedge.exe,16599
1,svchost.exe,15294
2,git.exe,11544
3,conhost.exe,9234
4,backgroundtaskhost.exe,5278
5,systemd-cgroups-agent,3449
6,runtimebroker.exe,2935
7,taskhostw.exe,2595
8,mscorsvw.exe,2559
9,searchfilterhost.exe,2232


Malicious label counts:


...

,alert_label,n_nodes
0,unlabelled,109718
1,benign,7203
2,artifact,249
3,malicious,163


Attack-label counts:


,is_attack_label,n_nodes
0,False,116921
1,True,412


Saved node times: 117,333 rows -> /home/asmi28/python_code/data/CARBANAK/processed/carbanak_node_times.csv


,process_guid,time,time_source
0,7dmf69pk-0c48813d-00000300-00000000-1da92af6fe...,1713807.777,child_event
1,7dmf69pk-0c48813d-00001234-00000000-1da92af715...,1713807.169,child_event
2,7dmf69pk-0c48813d-00000a04-00000000-1da94db181...,1713815.087,child_event
3,7dmf69pk-0c48813d-000028ac-00000000-1da94db182...,1713807.249,child_event
4,7dmf69pk-0c48813d-00000910-00000374-1da94daa3f...,1713807.189,child_event
5,7dmf69pk-0c48813d-000012b0-00000374-1da94daa4d...,1713807.189,child_event
6,7dmf69pk-0c48813d-000016e8-00000000-1da94db288...,1713815.087,child_event
7,7dmf69pk-0c48813d-00002278-00000000-1da94db289...,1713807.249,child_event
8,7dmf69pk-0c48813d-00000b00-00000000-1da94db29a...,1713815.087,child_event
9,7dmf69pk-0c48813d-00002b10-00000000-1da94db29a...,1713807.249,child_event


## Select connected-subgraph subsample

In [9]:

components = list(nx.weakly_connected_components(G))
if not components:
    raise RuntimeError("The graph has no weakly connected components.")

node_to_component = {}
component_node_counts = []
for cid, nodes in enumerate(components):
    component_node_counts.append((cid, len(nodes)))
    for node in nodes:
        node_to_component[node] = cid

df["_component_id_parent"] = df["parent_guid"].map(node_to_component)
df["_component_id"] = df["process_guid"].map(node_to_component)
if df["_component_id_parent"].isna().any() or df["_component_id"].isna().any():
    raise RuntimeError("At least one CSV endpoint was not found in the NetworkX graph.")
if not (df["_component_id_parent"] == df["_component_id"]).all():
    raise RuntimeError("An edge endpoint pair crosses weak components, which should be impossible.")

row_counts = df["_component_id"].value_counts().rename("n_rows")
node_counts = pd.Series(dict(component_node_counts), name="n_nodes")
component_summary = pd.concat([node_counts, row_counts], axis=1).fillna(0).astype({"n_nodes": int, "n_rows": int})
component_summary.index.name = "component_id"
component_summary = component_summary.reset_index()
# Attach external label summaries to components. These are used by EDA notebooks
# to preferentially select components containing malicious process GUIDs.
label_by_node = node_label_df.set_index("process_guid") if "node_label_df" in globals() else pd.DataFrame()
malicious_count_by_component = {}
attack_count_by_component = {}
for cid, nodes in enumerate(components):
    node_strings = [str(n) for n in nodes]
    if not label_by_node.empty:
        subset = label_by_node.reindex(node_strings)
        malicious_count_by_component[cid] = int(subset.get("is_malicious", pd.Series(False, index=subset.index)).fillna(False).astype(bool).sum())
        attack_count_by_component[cid] = int(subset.get("is_attack_label", pd.Series(False, index=subset.index)).fillna(False).astype(bool).sum())
    else:
        malicious_count_by_component[cid] = 0
        attack_count_by_component[cid] = 0

component_summary["n_malicious_nodes"] = component_summary["component_id"].map(malicious_count_by_component).fillna(0).astype(int)
component_summary["n_attack_label_nodes"] = component_summary["component_id"].map(attack_count_by_component).fillna(0).astype(int)
component_summary["estimated_csv_bytes"] = component_summary["n_rows"] * max(1, int(PROCESS_EDGES_CSV.stat().st_size / max(len(df), 1)))
if PREFER_MALICIOUS_COMPONENTS and component_summary["n_malicious_nodes"].sum() > 0:
    component_summary = component_summary.sort_values(
        ["n_malicious_nodes", "n_attack_label_nodes", "n_nodes", "n_rows"],
        ascending=False,
    )
elif PREFER_MALICIOUS_COMPONENTS and component_summary["n_attack_label_nodes"].sum() > 0:
    component_summary = component_summary.sort_values(
        ["n_attack_label_nodes", "n_nodes", "n_rows"],
        ascending=False,
    )
else:
    component_summary = component_summary.sort_values(["n_nodes", "n_rows"], ascending=False)
component_summary.to_csv(COMPONENT_SUMMARY_CSV, index=False)

candidate_summary = component_summary[component_summary["n_nodes"] >= MIN_COMPONENT_NODES_FOR_SUBSAMPLE].copy()
if REQUIRE_MIN_SUBSAMPLE_COMPONENTS and len(candidate_summary) < MIN_SUBSAMPLE_COMPONENTS:
    print("Largest components:")
    display(component_summary.head(50))
    raise RuntimeError(
        f"Only {len(candidate_summary)} components have at least {MIN_COMPONENT_NODES_FOR_SUBSAMPLE} nodes, "
        f"but MIN_SUBSAMPLE_COMPONENTS={MIN_SUBSAMPLE_COMPONENTS}. This usually means the edge extraction/GUID "
        "normalization still is not connecting process chains."
    )

if COMPONENT_ORDER == "largest_first":
    ordered_component_ids = candidate_summary["component_id"].tolist()
elif COMPONENT_ORDER == "random":
    ordered_component_ids = candidate_summary["component_id"].tolist()
    random.Random(RANDOM_SEED).shuffle(ordered_component_ids)
else:
    raise ValueError(f"Unknown COMPONENT_ORDER: {COMPONENT_ORDER!r}")

selected_ids = []
selected_bytes = 0
size_lookup = dict(zip(candidate_summary["component_id"], candidate_summary["estimated_csv_bytes"]))

for cid in ordered_component_ids:
    if MAX_SUBSAMPLE_COMPONENTS is not None and len(selected_ids) >= MAX_SUBSAMPLE_COMPONENTS:
        break
    selected_ids.append(cid)
    selected_bytes += int(size_lookup.get(cid, 0))
    if len(selected_ids) >= MIN_SUBSAMPLE_COMPONENTS and selected_bytes >= TARGET_SUBSAMPLE_BYTES:
        break

if not selected_ids:
    raise RuntimeError("Subsample selection chose no components.")

selected_ids_set = set(selected_ids)
sub_df = df[df["_component_id"].isin(selected_ids_set)][OUTPUT_COLUMNS].copy()
sub_df.to_csv(SUBSAMPLE_EDGES_CSV, index=False)

selected_component_summary = component_summary[component_summary["component_id"].isin(selected_ids_set)].copy()
selected_component_summary.to_csv(SUBSAMPLE_COMPONENTS_CSV, index=False)

G_sub = nx.from_pandas_edgelist(
    sub_df,
    source="parent_guid",
    target="process_guid",
    create_using=nx.DiGraph()
)
with open(SUBSAMPLE_GRAPH_PKL, "wb") as f:
    pickle.dump(G_sub, f, protocol=pickle.HIGHEST_PROTOCOL)

sub_nodes = {str(n) for n in G_sub.nodes()}
node_label_df[node_label_df["process_guid"].astype(str).isin(sub_nodes)].to_csv(SUBSAMPLE_NODE_LABELS_CSV, index=False)
node_time_df[node_time_df["process_guid"].astype(str).isin(sub_nodes)].to_csv(SUBSAMPLE_NODE_TIMES_CSV, index=False)

print(f"Candidate components >= {MIN_COMPONENT_NODES_FOR_SUBSAMPLE} nodes: {len(candidate_summary):,}")
print(f"Selected weak components:  {len(selected_ids):,}")
print(f"Subsample rows:            {len(sub_df):,}")
print(f"Subsample graph nodes:     {G_sub.number_of_nodes():,}")
print(f"Subsample graph edges:     {G_sub.number_of_edges():,}")
print(f"Subsample CSV size:        {SUBSAMPLE_EDGES_CSV.stat().st_size / (1024**2):.2f} MB")
print("Saved subsample CSV to:", SUBSAMPLE_EDGES_CSV)
print("Saved subsample graph to:", SUBSAMPLE_GRAPH_PKL)
print("Saved component summary to:", COMPONENT_SUMMARY_CSV)
print("Saved subsample labels to:", SUBSAMPLE_NODE_LABELS_CSV)
display(selected_component_summary.head(30))


Candidate components >= 3 nodes: 310
Selected weak components:  310
Subsample rows:            267,281
Subsample graph nodes:     117,305
Subsample graph edges:     117,915
Subsample CSV size:        34.16 MB
Saved subsample CSV to: /home/asmi28/python_code/data/CARBANAK/processed/carbanak_process_edges_subsample.csv
Saved subsample graph to: /home/asmi28/python_code/data/CARBANAK/processed/carbanak_process_graph_subsample.pkl
Saved component summary to: /home/asmi28/python_code/data/CARBANAK/processed/carbanak_component_summary.csv
Saved subsample labels to: /home/asmi28/python_code/data/CARBANAK/processed/carbanak_node_labels_subsample.csv


,component_id,n_nodes,n_rows,n_malicious_nodes,n_attack_label_nodes,estimated_csv_bytes
272,272,8018,25264,37,47,3385376
137,137,260,687,21,21,92058
133,133,114,259,15,15,34706
233,233,5588,14722,12,29,1972748
320,320,6749,13709,11,11,1837006
65,65,101,193,7,7,25862
119,119,959,2968,6,10,397712
107,107,597,1548,6,7,207432
227,227,277,649,6,7,86966
225,225,435,1153,5,6,154502


## Load the saved subsample later

In [10]:

# Later use:
# import pickle
# import pandas as pd
# import networkx as nx
#
# sub_df = pd.read_csv(SUBSAMPLE_EDGES_CSV, dtype={"parent_guid": "string", "process_guid": "string"})
# sub_labels = pd.read_csv(SUBSAMPLE_NODE_LABELS_CSV)
# with open(SUBSAMPLE_GRAPH_PKL, "rb") as f:
#     G_sub = pickle.load(f)
#
# print(sub_df.shape)
# print(G_sub.number_of_nodes(), G_sub.number_of_edges())
# display(sub_labels.head())
